# Install

In [ ]:
!pip3 install transformers
!pip3 install datasets
!pip3 install evaluate

# GPU Check

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Apr 27 09:14:38 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   31C    P0    42W / 400W |      0MiB / 40960MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 89.6 gigabytes of available RAM

You are using a high-RAM runtime!


# Alpaca Few-Shot Test

In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
model_name = "beomi/KoAlpaca-Polyglot"

In [6]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# model
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

#### Predict

In [17]:
# new_doc = "일요일에 데이트 어때?💕	[계절밥상] 소중한 사람과 함께 하는 통 돼지구이 FLEX!✨\\ 만원 할인 쿠폰 받고 가요!"
new_doc = "짝짝짝 친구 맺으면 1,OOOP	정답만 맞히면 포인트 주는 이벤트!\\퀴즈 맞히고 1,OOOP 바로 받아가기"

In [18]:
text = f"""문장: '{new_doc}'\n 키워드 추출:"""

In [19]:
encoded = tokenizer(text)

sample = {k: torch.tensor([v]).to(device) for k, v in encoded.items()}
del sample['token_type_ids']
print(sample)

{'input_ids': tensor([[  461,   349,    29,   514,  2255,  2255,  2255,  2081,  3182,  1253,
           376,    15, 19018,  9782,   201,   371,  1477,   380, 21151,   378,
          4068,   458,   272,  3263,     4,    63, 23343, 21151,   283,   376,
            15, 19018,  9782,  1617,   719,  2633,   316,    10,   202,  8664,
          9190,    29]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [20]:
with torch.no_grad():
    pred = model.generate(
        **sample,
        penalty_alpha=0.6,
        top_k=5,
        max_length=512,
        eos_token_id=tokenizer.eos_token_id,
    )

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [21]:
print(tokenizer.decode(pred[0], skip_special_tokens=True))

문장: '짝짝짝 친구 맺으면 1,OOOP	정답만 맞히면 포인트 주는 이벤트!\퀴즈 맞히고 1,OOOP 바로 받아가기'
 키워드 추출: '친구맺기', '포인트', '이벤트'
